習題一：使用random模組，產生出一個介於0到1之間的小數$r$，使用while迴圈，計算出$r$的前二十位二進位表示$r = 0.b_1b_2b_3\cdots {b_{20}}_2$。輸出格式如附圖Exercise20261015.png

In [ ]:
import random

# 使用 random 模組產生一個介於 0 到 1 之間的小數 r
r = random.random()

print(f"假如 r = {r}")

# 初始化二進位小數的字串與計算變數
binary_digits = ""
temp_r = r
count = 0

# 使用 while 迴圈計算前 20 位二進位表示
while count < 20:
  temp_r *= 2
  bit = int(temp_r)
  binary_digits += str(bit)
  temp_r -= bit
  count += 1

# 輸出結果
print(f"r ≈ 0.{binary_digits}_2")

假如 r = 0.725030787641855
r ≈ 0.10111001100110111001_2


Python 的 `int` 型別採用 Arbitrary-Precision Integer（任意精度整數）設計，能夠精確表示任意大小的整數，其實作方式可參考 CPython 的 `PyLongObject`。與 C/C++ 的 `int` 或 `long long` 不同，Python 的整數不受固定 32 位元或64 位元限制，而是會依據數值大小自動配置更多記憶體，因此只要記憶體足夠，便能表示極大的整數。例如：
```python
n = 10**68
print(n)
```
上述程式可正常執行，表示 Python 能夠精確儲存$10^{68}$，即中文數字中的「無量大數」。


可利用下列程式觀察不同大小整數所需的位元數（bit）與記憶體空間：

```python
import sys
n = 10**68
print(f"bit_length = {n.bit_length()}")
print(f"memory = {sys.getsizeof(n)} bytes")
```

並查閱官方文件，了解：

- `bit_length()`
- `sys.getsizeof()`
- `PyLongObject`

三者之間的關係。

習題二：輸入一個正整數 \(n\)，其中$1 \le n \le 10^{68}$，請將 \(n\) 轉換為中文大寫數字表示法。需支援下列數字單位：萬、億、兆、京、垓、秭、穰、溝、澗、正、載、極、恆河沙、阿僧祇、那由他、不可思議、無量大數。比如輸入31415926，輸出參仟壹佰肆拾壹萬伍仟玖佰貳拾陸。

提示：思考利用字典，建立「數字」與「中文大寫數字」的對應關係。

In [6]:
def number_to_chinese(n_str):
  # 移除開頭可能多餘的零或空白
  n_str = n_str.strip().lstrip('0')
  if not n_str:
    return '零'

  units = [
      '',
      '萬',
      '億',
      '兆',
      '京',
      '垓',
      '秭',
      '穰',
      '溝',
      '澗',
      '正',
      '載',
      '極',
      '恆河沙',
      '阿僧祇',
      '那由他',
      '不可思議',
      '無量大數',
  ]
  chinese_digits = ['零', '一', '二', '三', '四', '五', '六', '七', '八', '九']

  length = len(n_str)
  groups = []
  for i in range(length, 0, -4):
    start = max(0, i - 4)
    groups.append(n_str[start:i])

  if len(groups) > len(units):
    return '數值過大，超出支援的單位範圍！'

  res_parts = []
  zero_flag = False

  for i, group in enumerate(groups):
    group_val = int(group)
    unit_name = units[i]

    if group_val == 0:
      zero_flag = True
      continue

    group_str = convert_four_digits(group, chinese_digits)

    if zero_flag and res_parts:
      res_parts.append('零')

    zero_flag = False
    res_parts.append(group_str + unit_name)

  result = ''.join(reversed(res_parts))
  return result


def convert_four_digits(s, chinese_digits):
  """輔助函數：將 1 到 4 位數的字串轉換為中文，正確處理中間的零"""
  s = s.zfill(4)
  sub_units = ['', '十', '百', '千']
  res = ''
  has_zero = False

  for j in range(4):
    digit = int(s[j])
    pos_unit = sub_units[3 - j]

    if digit == 0:
      has_zero = True
    else:
      if has_zero and res != '':
        res += '零'
        has_zero = False
      if digit == 1 and pos_unit == '十' and len(res) == 0 and j == 2:
        res += pos_unit
      else:
        res += chinese_digits[digit] + pos_unit

  return res.rstrip('零')


# --- 讓使用者輸入 N 並輸出結果 ---
if __name__ == '__main__':
  user_input = input('請輸入一個正整數 n (1 <= n <= 10^68): ')
  print(f'輸入：{user_input}')
  print(f'輸出：{number_to_chinese(user_input)}')

輸入：3275
輸出：三千二百七十五


習題三：請勿使用for、range，而是利用iterable、iter()、next()、while、StopIteration來設計九九乘法表。

In [7]:
# 建立外層被乘數的 iterable（1 到 9）
i_list = [1, 2, 3, 4, 5, 6, 7, 8, 9]

try:
  # 取得外層的 iterator
  i_iter = iter(i_list)

  while True:
    # 利用 next() 取得下一個被乘數 i，若結束則會引發 StopIteration
    i = next(i_iter)

    # 建立內層乘數的 iterable（1 到 9）
    j_list = [1, 2, 3, 4, 5, 6, 7, 8, 9]
    j_iter = iter(j_list)

    row_str = ""
    while True:
      try:
        # 利用 next() 取得下一個乘數 j
        j = next(j_iter)
        # 組合九九乘法表的一項，並加上間距
        row_str += f"{i}×{j}={i*j:2d}  "
      except StopIteration:
        # 當內層 iterator 結束時跳出內層 while 迴圈
        break

    # 輸出該行的九九乘法表結果
    print(row_str)

except StopIteration:
  # 當外層 iterator 結束時結束程式
  pass

1×1= 1  1×2= 2  1×3= 3  1×4= 4  1×5= 5  1×6= 6  1×7= 7  1×8= 8  1×9= 9  
2×1= 2  2×2= 4  2×3= 6  2×4= 8  2×5=10  2×6=12  2×7=14  2×8=16  2×9=18  
3×1= 3  3×2= 6  3×3= 9  3×4=12  3×5=15  3×6=18  3×7=21  3×8=24  3×9=27  
4×1= 4  4×2= 8  4×3=12  4×4=16  4×5=20  4×6=24  4×7=28  4×8=32  4×9=36  
5×1= 5  5×2=10  5×3=15  5×4=20  5×5=25  5×6=30  5×7=35  5×8=40  5×9=45  
6×1= 6  6×2=12  6×3=18  6×4=24  6×5=30  6×6=36  6×7=42  6×8=48  6×9=54  
7×1= 7  7×2=14  7×3=21  7×4=28  7×5=35  7×6=42  7×7=49  7×8=56  7×9=63  
8×1= 8  8×2=16  8×3=24  8×4=32  8×5=40  8×6=48  8×7=56  8×8=64  8×9=72  
9×1= 9  9×2=18  9×3=27  9×4=36  9×5=45  9×6=54  9×7=63  9×8=72  9×9=81  
